## Prompts

In [2]:
import difflib, json, re
from typing import Optional

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel
from pathlib import Path
from dotenv import load_dotenv
import os

In [6]:
from pathlib import Path
INTERIM = Path.cwd().parent / "data" / "interim"

In [7]:
# config
load_dotenv(Path.cwd().parent / ".env")
MODEL = "gemini-3.5-flash-lite"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [8]:
print(client.models.generate_content(model=MODEL, contents="hi, who are you?").text)

Hi there! I'm a large language model, trained by Google. Think of me as a digital assistant—here to help you answer questions, brainstorm ideas, write content, code, and much more. 

How can I help you today?


In [31]:
prompt_v1 = '''
            You are a natural language extraction system. Your task is to understand the user query and return a JSON in the following format:
            {
                "semantic": [],
                "titles": [],
                "filters": {
                                "max_pages": null,
                                "min_pages":null,
                                "min_year": null,
                                "author_name": null,
                                "exclude_books":null,
                                "exclude_authors":null
                           },
                "follow_ups": []
            }

            1. Semantic should contain the semantic and mood signals, like "feel-good", "happy-ending".
            2. Titles should contain the book titles mentioned, corrected to canonical spelling. If a title could plausibly be two different books, leave it out of titles and put one short clarifying question in 'follow_ups'.
            3. Extract the filters if mentioned, or return null.
            4. Follow ups should be empty unless a follow up is asked.
          '''

In [32]:
user_query = "hi, i want to read something like the litle prince. i am in mood of soemthing easy to read and a short story."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v1)).text)

```json
{
    "semantic": [
        "easy to read",
        "short story",
        "whimsical",
        "philosophical",
        "poetic"
    ],
    "titles": [],
    "filters": {
        "max_pages": 150,
        "min_pages": null,
        "min_year": null,
        "author_name": null,
        "exclude_books": null,
        "exclude_authors": null
    },
    "follow_ups": []
}
```


In [33]:
prompt_v2 = '''
            You are a natural language extraction system. Your task is to understand the user query and return a JSON in the following format:
            {
                "semantic": [],
                "titles": [],
                "filters": {
                                "max_pages": null,
                                "min_pages":null,
                                "min_year": null,
                                "author_name": null,
                                "exclude_books":null,
                                "exclude_authors":null
                           },
                "follow_ups": []
            }

            1. Semantic should contain the semantic and mood signals, like "feel-good", "happy-ending". Exclude title, number of pages or signals like long or short or author names.
            2. Titles should contain the book titles mentioned, corrected to canonical spelling. If a title could plausibly be two different books, leave it out of titles and put one short clarifying question in 'follow_ups'.
            3. Extract the filters if mentioned, or return null.
            4. Follow ups should be empty unless a follow up is asked.
            5. Length signals: if user query has "short" mentioned, convert it to 200 pages as max_pages. For long, convert min_pages to 250.
          '''

In [34]:
user_query = "hi, i want to read something like the litle prince. i am in mood of soemthing easy to read and a short story."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v2)).text)

```json
{
    "semantic": [
        "feel-good",
        "philosophical",
        "whimsical"
    ],
    "titles": [
        "The Little Prince"
    ],
    "filters": {
        "max_pages": 200,
        "min_pages": null,
        "min_year": null,
        "author_name": null,
        "exclude_books": null,
        "exclude_authors": null
    },
    "follow_ups": []
}
```


In [44]:
prompt_v3 = '''
            You are a natural language extraction system. Your task is to understand the user query and return a JSON in the following format:
            {
                "semantic": [],
                "titles": [],
                "filters": {
                                "max_pages": null,
                                "min_pages":null,
                                "min_year": null,
                                "author_name": null,
                                "exclude_books":null,
                                "exclude_authors":null
                           },
                "follow_ups": []
            }

            1. Semantic should contain the semantic and mood signals, like "feel-good", "happy-ending". Exclude title, number of pages or signals like long or short or author names. They should come from the user's words', not the semantic of the title. Leave blank if none exists.
            2. Titles should contain the book titles mentioned, corrected to canonical spelling. If a title could plausibly be two different books, leave it out of titles and put one short clarifying question in 'follow_ups'.
            3. Extract the filters if mentioned, or return null.
            4. Follow ups should be empty unless a follow up is asked.
            5. Length signals: if user query has "short" mentioned, convert it to 200 pages as max_pages. For long, convert min_pages to 250.
          '''

In [45]:
user_query = "hi, i want to read something like the litle prince. i am in mood of soemthing easy to read and a short story."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v3)).text)

```json
{
    "semantic": [
        "easy to read"
    ],
    "titles": [
        "The Little Prince"
    ],
    "filters": {
        "max_pages": 200,
        "min_pages": null,
        "min_year": null,
        "author_name": null,
        "exclude_books": null,
        "exclude_authors": null
    },
    "follow_ups": []
}
```


In [46]:
user_query = "hi, i want to read something like hamet."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v3)).text)

```json
{
    "semantic": [],
    "titles": [
        "Hamnet"
    ],
    "filters": {
        "max_pages": null,
        "min_pages": null,
        "min_year": null,
        "author_name": null,
        "exclude_books": null,
        "exclude_authors": null
    },
    "follow_ups": []
}
```


In [ ]:
prompt_v4 = '''
You are a natural language extraction system. Understand the user's query and return JSON in exactly this format:

{
"semantic": [],
"title_mentions": [],
"proposed_titles": [],
"filters": {
"max_pages": null,
"min_pages": null,
"min_year": null,
"author_name": null,
"exclude_books": null,
"exclude_authors": null
},
"follow_ups": []
}

Rules:

1. "semantic":

   * Extract only semantic, mood, style, theme, or reading-preference signals explicitly stated by the user.
   * Examples: "feel-good", "easy to read", "dark", "philosophical".
   * Do NOT infer semantic attributes from a mentioned book.

2. "title_mentions":

   * Copy book-title mentions approximately as the user typed them.
   * Preserve misspellings.
   * Example: "litle prince" -> "litle prince".
   * Example: "hamegt" -> "hamegt".

3. "proposed_titles":

   * For each title mention, provide your best guess at the canonical book title.
   * Correct obvious spelling mistakes when possible.
   * Do not assume the proposed title is guaranteed correct; a downstream catalog resolver will validate it.
   * Keep the same order as "title_mentions".

4. "filters":

   * Extract only filters explicitly stated or implied by the rules below.
   * Unspecified filters must be null.
   * "short" -> max_pages = 200.
   * "long" -> min_pages = 250.

5. "follow_ups":

   * Leave empty.
   * Do not ask clarification questions about ambiguous titles; downstream title resolution handles ambiguity.

Return only valid JSON.
'''

In [48]:
user_query = "hi, i want to read something like the litle prince. i am in mood of soemthing easy to read and a short story."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v4)).text)

{
"semantic": [
"easy to read"
],
"title_mentions": [
"the litle prince"
],
"proposed_titles": [
"The Little Prince"
],
"filters": {
"max_pages": 200,
"min_pages": null,
"min_year": null,
"author_name": null,
"exclude_books": null,
"exclude_authors": null
},
"follow_ups": []
}


In [49]:
user_query = "hi, i want to read something like hamet."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=prompt_v4)).text)

{
"semantic": [],
"title_mentions": [
"hamet"
],
"proposed_titles": [
"Hamnet"
],
"filters": {
"max_pages": null,
"min_pages": null,
"min_year": null,
"author_name": null,
"exclude_books": null,
"exclude_authors": null
},
"follow_ups": []
}


In [1]:
PROMPT = """
You are a natural language extraction system. Understand the user's query and return JSON in exactly this format:

{
   "semantic": [],
   "title_mentions": [],
   "proposed_titles": [],
   "filters": {
                  "max_pages": null,
                  "min_pages": null,
                  "min_year": null,
                  "author_name": null,
                  "exclude_books": null,
                  "exclude_authors": null
               }
}

Rules:

1. semantic: only mood, style, theme or reading-preference signals the user stated. Examples: "feel-good", "easy to read", "dark". Never infer these from a book the user mentions. Exclude titles, author names and numbers.
2. title_mentions: book titles as the user typed them, misspellings preserved. "litle prince" -> "litle prince".
3. proposed_titles: your best guess at the canonical title for each mention, same order. A downstream catalog resolver validates these.
4. filters: only what the user stated. Unspecified filters are null. "short" -> max_pages 200. "long" -> min_pages 250.
5. Never ask about ambiguous titles. Title resolution happens downstream.
Return only valid JSON.
"""

In [5]:
user_query = "hi, i want to read something like the litle prince. i am in mood of soemthing easy to read and a short story."

print(client.models.generate_content(model=MODEL, contents=user_query, config=types.GenerateContentConfig(system_instruction=PROMPT)).text)

{
   "semantic": [
      "easy to read",
      "short story"
   ],
   "title_mentions": [
      "litle prince"
   ],
   "proposed_titles": [
      "The Little Prince"
   ],
   "filters": {
      "max_pages": 200,
      "min_pages": null,
      "min_year": null,
      "author_name": null,
      "exclude_books": null,
      "exclude_authors": null
   }
}
